In [1]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

from src.preprocessing import (
    separate_features_target,
    split_dataset
)

# Locate project root
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "default of credit card clients.xls"
MODEL_DIR = PROJECT_ROOT / "models"

TARGET_COLUMN = "default payment next month"

MODEL_FILES = {
    "Logistic Regression": "logistic_regression.pkl",
    "Decision Tree": "decision_tree.pkl",
    "KNN": "knn.pkl",
    "Gaussian Naive Bayes": "gaussian_naive_bayes.pkl",
    "SVM": "svm.pkl"
}

print("Project root:", PROJECT_ROOT)
print("Dataset exists:", DATA_PATH.exists())
print("Models directory exists:", MODEL_DIR.exists())

Project root: /Users/arinirsm/PycharmProjects/BITS-MTech-AIML-ML-Assignment-2
Dataset exists: True
Models directory exists: True


In [2]:
df = pd.read_excel(DATA_PATH, header=1)

print("DATASET OVERVIEW")
print("-" * 50)

print("Number of instances:", df.shape[0])
print("Number of columns:", df.shape[1])
print("Number of input features:", df.shape[1] - 1)

print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

/var/folders/wr/w2ybthrx1p7_7dwz41sqszz80000gn/T/ipykernel_99820/1014938119.py:1: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  df = pd.read_excel(DATA_PATH, header=1)


DATASET OVERVIEW
--------------------------------------------------
Number of instances: 30000
Number of columns: 25
Number of input features: 24

Column names:
['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default payment next month']

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   ID                          30000 non-null  int64
 1   LIMIT_BAL                   30000 non-null  int64
 2   SEX                         30000 non-null  int64
 3   EDUCATION                   30000 non-null  int64
 4   MARRIAGE                    30000 non-null  int64
 5   AGE                         30000 no

In [3]:
print("DATA QUALITY CHECKS")
print("-" * 50)

print("Total missing values:", df.isnull().sum().sum())
print("Total duplicate records:", df.duplicated().sum())

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nTarget distribution:")
target_counts = df[TARGET_COLUMN].value_counts().sort_index()

target_distribution = pd.DataFrame({
    "Count": target_counts,
    "Percentage": target_counts / len(df) * 100
})

target_distribution.index = [
    "No Default (0)",
    "Default (1)"
]

display(target_distribution.round(2))

DATA QUALITY CHECKS
--------------------------------------------------
Total missing values: 0
Total duplicate records: 0

Missing values per column:
ID                            0
LIMIT_BAL                     0
SEX                           0
EDUCATION                     0
MARRIAGE                      0
AGE                           0
PAY_0                         0
PAY_2                         0
PAY_3                         0
PAY_4                         0
PAY_5                         0
PAY_6                         0
BILL_AMT1                     0
BILL_AMT2                     0
BILL_AMT3                     0
BILL_AMT4                     0
BILL_AMT5                     0
BILL_AMT6                     0
PAY_AMT1                      0
PAY_AMT2                      0
PAY_AMT3                      0
PAY_AMT4                      0
PAY_AMT5                      0
PAY_AMT6                      0
default payment next month    0
dtype: int64

Target distribution:


,Count,Percentage
No Default (0),23364,77.88
Default (1),6636,22.12


In [4]:
X, y = separate_features_target(df)

print("FEATURE AND TARGET DETAILS")
print("-" * 50)

print("Input feature count:", X.shape[1])
print("Target variable:", TARGET_COLUMN)

print("\nInput features:")
print(X.columns.tolist())

print("\nFeature dimensions:", X.shape)
print("Target dimensions:", y.shape)

print("\nTarget classes:")
print(y.value_counts().sort_index())

FEATURE AND TARGET DETAILS
--------------------------------------------------
Input feature count: 23
Target variable: default payment next month

Input features:
['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']

Feature dimensions: (30000, 23)
Target dimensions: (30000,)

Target classes:
default payment next month
0    23364
1     6636
Name: count, dtype: int64


In [5]:
X_train, X_test, y_train, y_test = split_dataset(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("TRAIN-TEST SPLIT")
print("-" * 50)

print("Training instances:", len(X_train))
print("Testing instances:", len(X_test))

print("Training features:", X_train.shape[1])
print("Testing features:", X_test.shape[1])

print("\nTraining target distribution:")
print(y_train.value_counts().sort_index())

print("\nTesting target distribution:")
print(y_test.value_counts().sort_index())

print("\nTraining percentage:", round(len(X_train) / len(df) * 100, 2))
print("Testing percentage:", round(len(X_test) / len(df) * 100, 2))

TRAIN-TEST SPLIT
--------------------------------------------------
Training instances: 24000
Testing instances: 6000
Training features: 23
Testing features: 23

Training target distribution:
default payment next month
0    18691
1     5309
Name: count, dtype: int64

Testing target distribution:
default payment next month
0    4673
1    1327
Name: count, dtype: int64

Training percentage: 80.0
Testing percentage: 20.0


In [6]:
models = {}

for model_name, filename in MODEL_FILES.items():

    model_path = MODEL_DIR / filename

    models[model_name] = joblib.load(model_path)

    print(f"Loaded: {model_name} -> {filename}")

print("\nTotal models loaded:", len(models))

Loaded: Logistic Regression -> logistic_regression.pkl
Loaded: Decision Tree -> decision_tree.pkl
Loaded: KNN -> knn.pkl
Loaded: Gaussian Naive Bayes -> gaussian_naive_bayes.pkl
Loaded: SVM -> svm.pkl

Total models loaded: 5


In [7]:
results = {}
predictions = {}
probabilities = {}

for model_name, model in models.items():

    # Predictions using the model's default threshold
    y_pred = model.predict(X_test)

    # Find probability corresponding to positive class (1)
    model_probabilities = model.predict_proba(X_test)

    positive_class_index = list(model.classes_).index(1)

    y_prob = model_probabilities[:, positive_class_index]

    # Store predictions and probabilities
    predictions[model_name] = y_pred
    probabilities[model_name] = y_prob

    # Calculate evaluation metrics
    results[model_name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob),
        "Precision": precision_score(
            y_test, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_test, y_pred, zero_division=0
        ),
        "F1 Score": f1_score(
            y_test, y_pred, zero_division=0
        ),
        "MCC": matthews_corrcoef(y_test, y_pred)
    }

metrics_df = pd.DataFrame(results).T

display(metrics_df.round(4))

,Accuracy,AUC,Precision,Recall,F1 Score,MCC
Logistic Regression,0.8088,0.7100,0.6923,0.2442,0.3610,0.3302
Decision Tree,0.7153,0.6016,0.3669,0.3956,0.3807,0.1965
KNN,0.7948,0.7040,0.5561,0.3587,0.4361,0.3292
Gaussian Naive Bayes,0.4163,0.6516,0.2495,0.8161,0.3821,0.1104
SVM,0.8080,0.6927,0.6834,0.2457,0.3614,0.3273


In [8]:
for model_name in models:

    print("\n" + "=" * 65)
    print(model_name.upper())
    print("=" * 65)

    print(
        classification_report(
            y_test,
            predictions[model_name],
            labels=[0, 1],
            target_names=["No Default", "Default"],
            digits=4,
            zero_division=0
        )
    )


LOGISTIC REGRESSION
              precision    recall  f1-score   support

  No Default     0.8187    0.9692    0.8876      4673
     Default     0.6923    0.2442    0.3610      1327

    accuracy                         0.8088      6000
   macro avg     0.7555    0.6067    0.6243      6000
weighted avg     0.7907    0.8088    0.7711      6000


DECISION TREE
              precision    recall  f1-score   support

  No Default     0.8245    0.8061    0.8152      4673
     Default     0.3669    0.3956    0.3807      1327

    accuracy                         0.7153      6000
   macro avg     0.5957    0.6009    0.5980      6000
weighted avg     0.7233    0.7153    0.7191      6000


KNN
              precision    recall  f1-score   support

  No Default     0.8346    0.9187    0.8746      4673
     Default     0.5561    0.3587    0.4361      1327

    accuracy                         0.7948      6000
   macro avg     0.6953    0.6387    0.6554      6000
weighted avg     0.7730    0.7948

In [9]:
confusion_matrices = {}

for model_name in models:

    cm = confusion_matrix(
        y_test,
        predictions[model_name],
        labels=[0, 1]
    )

    confusion_matrices[model_name] = cm

    tn, fp, fn, tp = cm.ravel()

    print("\n", model_name)
    print("-" * 45)

    print("True Negatives :", tn)
    print("False Positives:", fp)
    print("False Negatives:", fn)
    print("True Positives :", tp)

    print("\nConfusion Matrix:")
    display(
        pd.DataFrame(
            cm,
            index=["Actual No Default", "Actual Default"],
            columns=["Predicted No Default", "Predicted Default"]
        )
    )


 Logistic Regression
---------------------------------------------
True Negatives : 4529
False Positives: 144
False Negatives: 1003
True Positives : 324

Confusion Matrix:


,Predicted No Default,Predicted Default
Actual No Default,4529,144
Actual Default,1003,324



 Decision Tree
---------------------------------------------
True Negatives : 3767
False Positives: 906
False Negatives: 802
True Positives : 525

Confusion Matrix:


,Predicted No Default,Predicted Default
Actual No Default,3767,906
Actual Default,802,525



 KNN
---------------------------------------------
True Negatives : 4293
False Positives: 380
False Negatives: 851
True Positives : 476

Confusion Matrix:


,Predicted No Default,Predicted Default
Actual No Default,4293,380
Actual Default,851,476



 Gaussian Naive Bayes
---------------------------------------------
True Negatives : 1415
False Positives: 3258
False Negatives: 244
True Positives : 1083

Confusion Matrix:


,Predicted No Default,Predicted Default
Actual No Default,1415,3258
Actual Default,244,1083



 SVM
---------------------------------------------
True Negatives : 4522
False Positives: 151
False Negatives: 1001
True Positives : 326

Confusion Matrix:


,Predicted No Default,Predicted Default
Actual No Default,4522,151
Actual Default,1001,326


In [10]:
print("BEST MODEL BY METRIC")
print("-" * 50)

for metric in metrics_df.columns:

    best_model = metrics_df[metric].idxmax()
    best_score = metrics_df[metric].max()

    print(
        f"{metric}: {best_model} ({best_score:.4f})"
    )

BEST MODEL BY METRIC
--------------------------------------------------
Accuracy: Logistic Regression (0.8088)
AUC: Logistic Regression (0.7100)
Precision: Logistic Regression (0.6923)
Recall: Gaussian Naive Bayes (0.8161)
F1 Score: KNN (0.4361)
MCC: Logistic Regression (0.3302)


In [11]:
class_performance = []

for model_name in models:

    report = classification_report(
        y_test,
        predictions[model_name],
        labels=[0, 1],
        target_names=["No Default", "Default"],
        output_dict=True,
        zero_division=0
    )

    class_performance.append({
        "Model": model_name,
        "Default Precision": report["Default"]["precision"],
        "Default Recall": report["Default"]["recall"],
        "Default F1": report["Default"]["f1-score"],
        "No Default Precision": report["No Default"]["precision"],
        "No Default Recall": report["No Default"]["recall"],
        "No Default F1": report["No Default"]["f1-score"]
    })

class_performance_df = pd.DataFrame(class_performance)

display(class_performance_df.round(4))

,Model,Default Precision,Default Recall,Default F1,No Default Precision,No Default Recall,No Default F1
0,Logistic Regression,0.6923,0.2442,0.3610,0.8187,0.9692,0.8876
1,Decision Tree,0.3669,0.3956,0.3807,0.8245,0.8061,0.8152
2,KNN,0.5561,0.3587,0.4361,0.8346,0.9187,0.8746
3,Gaussian Naive Bayes,0.2495,0.8161,0.3821,0.8529,0.3028,0.4469
4,SVM,0.6834,0.2457,0.3614,0.8188,0.9677,0.8870


In [12]:
OUTPUT_DIR = PROJECT_ROOT / "notebooks"

metrics_df.to_csv(
    OUTPUT_DIR / "model_evaluation_metrics.csv"
)

class_performance_df.to_csv(
    OUTPUT_DIR / "class_performance_comparison.csv",
    index=False
)

print("Evaluation results exported successfully.")

Evaluation results exported successfully.
